# 🗄️ Course 5 — Data Manipulation in SQL

> **Platform:** DataCamp | **Track:** Associate Data Analyst in SQL  
> **Tool:** PostgreSQL | **Database:** European Soccer Database (matches, leagues, teams, countries)

---

## 📋 About This Course

This course covers advanced SQL techniques for manipulating and transforming data. Using the European Soccer Database (12,800+ matches from 11 countries, 2011-2015), it progresses from CASE statements through subqueries, correlated queries, CTEs, and window functions.

---

## 📚 Table of Contents

| Chapter | Topic |
|---------|-------|
| Chapter 1 | CASE Statements |
| Chapter 2 | Short and Simple Subqueries |
| Chapter 3 | Correlated Queries, Nested Queries & CTEs |
| Chapter 4 | Window Functions |

---

## 📌 Chapter 1 — CASE Statements

---

### Basic CASE Statement
Identify matches in Germany as FC Bayern Munich, FC Schalke 04, or Other as the home team.

In [ ]:
SELECT 
	CASE WHEN hometeam_id = 10189 THEN 'FC Schalke 04'
	     WHEN hometeam_id = 9823 THEN 'FC Bayern Munich'
	     ELSE 'Other' END AS home_team,
	COUNT(id) AS total_matches
FROM matches_germany
GROUP BY home_team;

### CASE Comparing Column Values
Identify Barcelona's 2011/2012 season matches as home wins, losses, or ties.

In [ ]:
SELECT 
	date,
	CASE WHEN home_goal > away_goal THEN 'Home win!'
	     WHEN home_goal < away_goal THEN 'Home loss :(' 
	     ELSE 'Tie' END AS outcome
FROM matches_spain;

### CASE for Away Team
Identify Barcelona's results as the away team, joining the home team's name.

In [ ]:
SELECT  
	m.date,
	t.team_long_name AS opponent,
	CASE WHEN home_goal < away_goal THEN 'Barcelona win!'
	     WHEN home_goal > away_goal THEN 'Barcelona loss :(' 
	     ELSE 'Tie' END AS outcome
FROM matches_spain AS m
LEFT JOIN teams_spain AS t 
ON m.hometeam_id = t.team_api_id
WHERE m.awayteam_id = 8634;

### El Clásico: Barcelona vs Real Madrid
Categorize matches between Barcelona and Real Madrid by winner using multiple CASE statements.

In [ ]:
SELECT 
	date,
	CASE WHEN hometeam_id = 8634 THEN 'FC Barcelona' 
	     ELSE 'Real Madrid CF' END AS home,
	CASE WHEN awayteam_id = 8634 THEN 'FC Barcelona' 
	     ELSE 'Real Madrid CF' END AS away,
	CASE WHEN home_goal > away_goal AND hometeam_id = 8634 THEN 'Barcelona win!'
	     WHEN home_goal < away_goal AND awayteam_id = 8633 THEN 'Real Madrid win!'
	     ELSE 'Tie!' END AS outcome
FROM matches_spain
WHERE hometeam_id = 8634 AND awayteam_id = 8633;

### Filtering with CASE in WHERE
Generate all matches won by Bologna (id=9857) using CASE as a filter.

In [ ]:
SELECT season, date, home_goal, away_goal
FROM matches_italy
WHERE 
	CASE WHEN hometeam_id = 9857 AND home_goal > away_goal THEN 'Bologna Win'
	     WHEN awayteam_id = 9857 AND away_goal > home_goal THEN 'Bologna Win' 
	     END IS NOT NULL;

### COUNT with CASE WHEN
Count matches played per country in the 2012/2013 and 2013/2014 seasons as separate columns.

In [ ]:
SELECT 
	c.name AS country,
	COUNT(CASE WHEN m.season = '2012/2013' THEN m.id END) AS matches_2012_2013,
	COUNT(CASE WHEN m.season = '2013/2014' THEN m.id END) AS matches_2013_2014
FROM country AS c
LEFT JOIN match AS m ON c.id = m.country_id
GROUP BY country;

### SUM with CASE WHEN
Calculate total home and away goals scored by Real Sociedad (id=8560) per season.

In [ ]:
SELECT season,
	SUM(CASE WHEN hometeam_id = 8560 THEN home_goal END) AS home_goals,
	SUM(CASE WHEN awayteam_id = 8560 THEN away_goal END) AS away_goals
FROM match
GROUP BY season;

### AVG with CASE WHEN — Calculating Fractions
Calculate the fraction of tied games per country in 2013/2014 and 2014/2015 seasons.  
> **Note:** `<>` means "not equal to" (same as `!=`)

In [ ]:
SELECT 
	c.name AS country,
	AVG(CASE WHEN m.season='2013/2014' AND m.home_goal = m.away_goal THEN 1
		     WHEN m.season='2013/2014' AND m.home_goal <> m.away_goal THEN 0
		     END) AS ties_2013_2014,
	AVG(CASE WHEN m.season='2014/2015' AND m.home_goal = m.away_goal THEN 1
		     WHEN m.season='2014/2015' AND m.home_goal != m.away_goal THEN 0
		     END) AS ties_2014_2015
FROM country AS c
LEFT JOIN matches AS m ON c.id = m.country_id
GROUP BY country;

---

## 📌 Chapter 2 — Short and Simple Subqueries

---

### Scalar Subquery in WHERE
Find matches where total goals exceed 3× the season average.

In [ ]:
SELECT date, home_goal, away_goal
FROM matches_2013_2014
WHERE (home_goal + away_goal) > 
       (SELECT AVG(home_goal + away_goal) * 3
        FROM matches_2013_2014);

### Subquery with a List
Find teams that never played a home game (not in hometeam_id list).

In [ ]:
SELECT team_long_name, team_short_name
FROM team 
WHERE team_api_id NOT IN 
     (SELECT DISTINCT hometeam_id FROM match);

### Complex Subquery Condition
Find teams that scored 8 or more goals in a home match.

In [ ]:
SELECT team_long_name, team_short_name
FROM team
WHERE team_api_id IN
	  (SELECT hometeam_id
       FROM match
       WHERE home_goal >= 8);

### Subquery in FROM
Count matches with 10+ total goals per country using a subquery in FROM.

In [ ]:
SELECT
	c.name AS country_name,
	COUNT(sub.id) AS matches
FROM country AS c
INNER JOIN (SELECT country_id, id 
            FROM match
            WHERE (home_goal + away_goal) >= 10) AS sub
ON c.id = sub.country_id
GROUP BY country_name;

### Building on FROM Subquery
Get full details (date, goals) for matches with 10+ total goals.

In [ ]:
SELECT country, date, home_goal, away_goal
FROM 
	(SELECT 
		 c.name AS country, m.date, m.home_goal, m.away_goal,
		 (m.home_goal + m.away_goal) AS total_goals
	 FROM match AS m
	 LEFT JOIN country AS c ON m.country_id = c.id) AS subq
WHERE total_goals >= 10;

### Subquery in SELECT
Add the overall season average alongside each league's average goals.

In [ ]:
SELECT 
	l.name AS league,
	ROUND(AVG(m.home_goal + m.away_goal), 2) AS avg_goals,
	(SELECT ROUND(AVG(home_goal + away_goal), 2) 
	 FROM match WHERE season = '2013/2014') AS overall_avg
FROM league AS l
LEFT JOIN match AS m ON l.country_id = m.country_id
WHERE season = '2013/2014'
GROUP BY l.name;

### Subquery in SELECT — Calculating Differences
Calculate each league's average goals minus the overall average for 2013/2014.

In [ ]:
SELECT
	l.name AS league,
	ROUND(AVG(m.home_goal + m.away_goal), 2) AS avg_goals,
	ROUND(AVG(m.home_goal + m.away_goal) - 
		(SELECT AVG(home_goal + away_goal)
		 FROM match WHERE season = '2013/2014'), 2) AS diff
FROM league AS l
LEFT JOIN match AS m ON l.country_id = m.country_id
WHERE season = '2013/2014'
GROUP BY l.name;

### Subqueries Everywhere
Combine SELECT, FROM, and WHERE subqueries to analyze average goals per match stage in 2012/2013.

In [ ]:
-- Step 1: SELECT + WHERE subqueries
SELECT 
	m.stage,
	ROUND(AVG(m.home_goal + m.away_goal), 2) AS avg_goals,
	ROUND((SELECT AVG(home_goal + away_goal) 
	       FROM match WHERE season = '2012/2013'), 2) AS overall
FROM match AS m
WHERE season = '2012/2013'
GROUP BY stage;

In [ ]:
-- Step 2: Add FROM subquery — filter stages above overall average
SELECT stage, ROUND(s.avg_goals, 2) AS avg_goals
FROM 
	(SELECT stage, AVG(home_goal + away_goal) AS avg_goals
	 FROM match WHERE season = '2012/2013'
	 GROUP BY stage) AS s
WHERE s.avg_goals > (SELECT AVG(home_goal + away_goal) 
					 FROM match WHERE season = '2012/2013');

In [ ]:
-- Step 3: Full query with SELECT, FROM, and WHERE subqueries
SELECT 
	stage,
	ROUND(s.avg_goals, 2) AS avg_goal,
	(SELECT AVG(home_goal + away_goal) 
	 FROM match WHERE season = '2012/2013') AS overall_avg
FROM 
	(SELECT stage, AVG(home_goal + away_goal) AS avg_goals
	 FROM match WHERE season = '2012/2013'
	 GROUP BY stage) AS s
WHERE s.avg_goals > (SELECT AVG(home_goal + away_goal) 
					 FROM match WHERE season = '2012/2013');

---

## 📌 Chapter 3 — Correlated Queries, Nested Queries & CTEs

---

### Correlated Subquery
Find matches where total goals exceed 3× the average for that country.

In [ ]:
SELECT main.country_id, main.date, main.home_goal, main.away_goal
FROM match AS main
WHERE (home_goal + away_goal) > 
        (SELECT AVG((sub.home_goal + sub.away_goal) * 3)
         FROM match AS sub
         WHERE main.country_id = sub.country_id);

### Correlated Subquery with Multiple Conditions
Find the highest-scoring match per country, per season.

In [ ]:
SELECT main.country_id, main.date, main.home_goal, main.away_goal
FROM match AS main
WHERE (home_goal + away_goal) = 
        (SELECT MAX(sub.home_goal + sub.away_goal)
         FROM match AS sub
         WHERE main.country_id = sub.country_id
               AND main.season = sub.season);

### Nested Subquery
Compare each season's max goals against the England Premier League max for the same season.

In [ ]:
SELECT 
    season,
    MAX(home_goal + away_goal) AS max_goals,
    (SELECT MAX(home_goal + away_goal) 
     FROM match 
     WHERE season = main.season
     AND country_id IN 
        (SELECT country_id FROM league 
         WHERE name = 'England Premier League')) AS pl_max_goals
FROM match AS main
GROUP BY season;

### Nested Subquery in FROM
Calculate the average number of high-scoring matches (5+ goals) per season, per country.

In [ ]:
SELECT
	c.name AS country,
	AVG(outer_s.matches) AS avg_seasonal_high_scores
FROM country AS c
LEFT JOIN (
  SELECT country_id, season, COUNT(id) AS matches
  FROM (
    SELECT country_id, season, id
	FROM match
	WHERE home_goal >= 5 OR away_goal >= 5) AS inner_s
  GROUP BY country_id, season) AS outer_s
ON c.id = outer_s.country_id
GROUP BY country;

### Clean Up with CTEs
Rewrite a subquery-based query using a CTE to count matches with 10+ goals per league.

In [ ]:
WITH match_list AS (
    SELECT country_id, id
    FROM match
    WHERE (home_goal + away_goal) >= 10)

SELECT l.name AS league, COUNT(match_list.id) AS matches
FROM league AS l
LEFT JOIN match_list ON l.id = match_list.country_id
GROUP BY l.name;

### Organizing with CTEs
Create a CTE joining match and league, then filter for games with 10+ total goals.

In [ ]:
WITH match_list AS (
    SELECT 
		l.name AS league, date, m.home_goal, m.away_goal,
		(m.home_goal + m.away_goal) AS total_goals
    FROM match AS m
    LEFT JOIN league AS l ON m.country_id = l.id)

SELECT league, date, home_goal, away_goal
FROM match_list
WHERE total_goals >= 10;

### CTE with Nested Subquery
Calculate average goals per league in August of the 2013/2014 season using a CTE with a nested subquery.

In [ ]:
WITH match_list AS (
    SELECT 
		country_id,
		(home_goal + away_goal) AS goals
    FROM match
    WHERE id IN (
       SELECT match.id FROM match
       WHERE season = '2013/2014' AND EXTRACT(MONTH FROM date) = 8))

SELECT l.name, AVG(goals)
FROM league AS l
LEFT JOIN match_list ON l.id = match_list.country_id
GROUP BY l.name;

### 📋 Summary — When to Use Each Technique

| Technique | Best For |
|-----------|----------|
| **Joins** | Combining 2+ tables, simple aggregations |
| **Correlated Subqueries** | Matching across columns, avoiding complex joins |
| **Nested Subqueries** | Multi-step transformations, complex calculations |
| **CTEs** | Organizing subqueries sequentially, reusability |

> **Q:** Which statement is FALSE — Correlated subqueries reduce query length AND improve run time?  
> **A:** FALSE — correlated subqueries run once per row, so they are **slower**, not faster.


### Get Team Names — 3 Methods
**Method 1:** Subqueries in FROM

In [ ]:
SELECT
	m.date,
	home.hometeam_name,
	away.awayteam_name,
	m.home_goal, m.away_goal
FROM match AS m
LEFT JOIN (
  SELECT match.id, team.team_long_name AS hometeam_name
  FROM match LEFT JOIN team ON match.hometeam_id = team.team_api_id) AS home
ON home.id = m.id
LEFT JOIN (
  SELECT match.id, team.team_long_name AS awayteam_name
  FROM match LEFT JOIN team ON match.awayteam_id = team.team_api_id) AS away
ON away.id = m.id;

**Method 2:** Correlated Subqueries in SELECT

In [ ]:
SELECT
    m.date,
    (SELECT team_long_name FROM team AS t
     WHERE t.team_api_id = m.hometeam_id) AS hometeam,
    (SELECT team_long_name FROM team AS t
     WHERE t.team_api_id = m.awayteam_id) AS awayteam,
    home_goal, away_goal
FROM match AS m;

**Method 3:** Common Table Expressions (CTEs)

In [ ]:
WITH home AS (
  SELECT m.id, m.date, t.team_long_name AS hometeam, m.home_goal
  FROM match AS m
  LEFT JOIN team AS t ON m.hometeam_id = t.team_api_id),

  away AS (
  SELECT m.id, m.date, t.team_long_name AS awayteam, m.away_goal
  FROM match AS m
  LEFT JOIN team AS t ON m.awayteam_id = t.team_api_id)

SELECT 
	home.date, home.hometeam, away.awayteam,
	home.home_goal, away.away_goal
FROM home
INNER JOIN away ON home.id = away.id;

---

## 📌 Chapter 4 — Window Functions

---

### OVER() — Basic Window Function
Add the overall average goals as a column in every row using OVER().

In [ ]:
SELECT 
	m.id, c.name AS country, m.season,
	m.home_goal, m.away_goal,
	AVG(m.home_goal + m.away_goal) OVER() AS overall_avg
FROM match AS m
LEFT JOIN country AS c ON m.country_id = c.id;

### RANK() — Ascending
Rank leagues by average total goals in the 2011/2012 season (lowest to highest).

In [ ]:
SELECT 
	l.name AS league,
	AVG(m.home_goal + m.away_goal) AS avg_goals,
	RANK() OVER(ORDER BY AVG(m.home_goal + m.away_goal)) AS league_rank
FROM league AS l
LEFT JOIN match AS m ON l.id = m.country_id
WHERE m.season = '2011/2012'
GROUP BY l.name
ORDER BY league_rank;

### RANK() — Descending
Rank leagues from highest to lowest average goals.

In [ ]:
SELECT 
	l.name AS league,
	AVG(m.home_goal + m.away_goal) AS avg_goals,
	RANK() OVER(ORDER BY AVG(m.home_goal + m.away_goal) DESC) AS league_rank
FROM league AS l
LEFT JOIN match AS m ON l.id = m.country_id
WHERE m.season = '2011/2012'
GROUP BY l.name
ORDER BY league_rank;

### PARTITION BY a Column
Calculate separate home/away goal averages for each season for Legia Warszawa (id=8673) matches.

In [ ]:
SELECT
	date, season, home_goal, away_goal,
	CASE WHEN hometeam_id = 8673 THEN 'home' ELSE 'away' END AS warsaw_location,
	AVG(home_goal) OVER(PARTITION BY season) AS season_homeavg,
	AVG(away_goal) OVER(PARTITION BY season) AS season_awayavg
FROM match
WHERE hometeam_id = 8673 OR awayteam_id = 8673
ORDER BY (home_goal + away_goal) DESC;

### PARTITION BY Multiple Columns
Partition home/away averages by season AND month for Legia Warszawa matches.

In [ ]:
SELECT 
	date, season, home_goal, away_goal,
	CASE WHEN hometeam_id = 8673 THEN 'home' ELSE 'away' END AS warsaw_location,
	AVG(home_goal) OVER(PARTITION BY season, EXTRACT(month FROM date)) AS season_mo_home,
	AVG(away_goal) OVER(PARTITION BY season, EXTRACT(month FROM date)) AS season_mo_away
FROM match
WHERE hometeam_id = 8673 OR awayteam_id = 8673
ORDER BY (home_goal + away_goal) DESC;

### Sliding Window — Running Total (Forward)
Calculate running total and average of home goals for FC Utrecht (id=9908) in 2011/2012.

In [ ]:
SELECT 
	date, home_goal, away_goal,
	SUM(home_goal) OVER(ORDER BY date 
	      ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS running_total,
	AVG(home_goal) OVER(ORDER BY date 
	      ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS running_avg
FROM match
WHERE hometeam_id = 9908 AND season = '2011/2012';

### Sliding Window — Running Total (Backward)
Calculate running total and average of away goals from current row to end (descending date).

In [ ]:
SELECT
	date, away_goal,
	SUM(away_goal) OVER(ORDER BY date DESC
	      ROWS BETWEEN CURRENT ROW AND UNBOUNDED FOLLOWING) AS running_total,
	AVG(away_goal) OVER(ORDER BY date DESC
	      ROWS BETWEEN CURRENT ROW AND UNBOUNDED FOLLOWING) AS running_avg
FROM match
WHERE awayteam_id = 9908 AND season = '2011/2012';

### Final Challenge — CTEs + CASE + RANK + Window Functions
Find Manchester United's losses in 2014/2015 EPL, ranked by goal difference.  
> **Note:** Rank 1 = worst loss (largest goal difference), last rank = narrowest loss.

In [ ]:
WITH home AS (
  SELECT m.id, t.team_long_name,
	  CASE WHEN m.home_goal > m.away_goal THEN 'MU Win'
		   WHEN m.home_goal < m.away_goal THEN 'MU Loss' 
  		   ELSE 'Tie' END AS outcome
  FROM match AS m
  LEFT JOIN team AS t ON m.hometeam_id = t.team_api_id),

  away AS (
  SELECT m.id, t.team_long_name,
	  CASE WHEN m.home_goal > m.away_goal THEN 'MU Loss'
		   WHEN m.home_goal < m.away_goal THEN 'MU Win' 
  		   ELSE 'Tie' END AS outcome
  FROM match AS m
  LEFT JOIN team AS t ON m.awayteam_id = t.team_api_id)

SELECT DISTINCT
    m.date,
    home.team_long_name AS home_team,
    away.team_long_name AS away_team,
    m.home_goal, m.away_goal,
    RANK() OVER(ORDER BY ABS(home_goal - away_goal) DESC) AS match_rank
FROM match AS m
LEFT JOIN home ON m.id = home.id
LEFT JOIN away ON m.id = away.id
WHERE m.season = '2014/2015'
      AND ((home.team_long_name = 'Manchester United' AND home.outcome = 'MU Loss')
      OR (away.team_long_name = 'Manchester United' AND away.outcome = 'MU Loss'));